# TakeMeter — training notebook

This notebook trains your classifier. It runs **on your own machine**, from
inside your project folder — no accounts, no upload, nothing to connect.

**Before you run anything:**

1. **Open it from your repo**, not from a copy somewhere else. Everything it
   writes lands next to your code so you can commit it normally.
2. **Activate your virtual environment first.** In VS Code, that's the kernel
   picker in the top right — choose the `.venv` inside this project. Getting
   this wrong is the single most common way the first cell fails.

How long it takes depends on what you're running on. Section 1 tells you, and
the first cell of section 4 prints a time as it goes. On a laptop CPU expect a
few minutes per seed; on a GPU, well under one.

---

### What's here

| Section | When | What it does |
|---|---|---|
| **0. Dataset** | units 5–6 | Data source, labeling, augmentation, and class distribution |
| **1. Setup** | unit 5 | Checks what you're running on and where |
| **2. Your labels** | unit 5 | Your label map, and reading `labels.csv` |
| **3. Split** | unit 5 | 70/15/15, with the seed as an editable constant |
| **4. Train** | unit 5 | The training run |
| **5. Results** | unit 5 | Writes `results.json` and `test_split.csv` |
| **6. Three seeds** | unit 6 | The same thing three times, on three splits |
| **7. Confusion matrix** | unit 6 | Which labels get mixed up, as a table |

## 0. Dataset

I selected Reddit's r/NeutralPolitics community and used a Kaggle Reddit dataset containing public posts from that subreddit. My selected dataset, `data/reditt_neutral_politics_labled_sorted.csv`, contains 200 posts across four topic categories: Executive Branch, Legal and Investigations, Foreign Policy, and Elections.

I assigned topic labels, also had Claude classify the posts, and then manually reviewed and corrected the labels. The dataset contains three columns:

- **`text`:** The post text supplied to the classifier.
- **`label`:** The topic category used as the target for training and evaluation.
- **`note`:** My observations about ambiguous posts or possible misclassification. These notes are excluded from the model's inputs.

The exact legal-category label in the CSV and this notebook is `Legal and investigations`. All four labels in the current `labels.csv` have no leading or trailing whitespace.

### Original dataset distribution

| Label | Posts | Share |
|---|---:|---:|
| Executive Branch | 97 | 48.5% |
| Legal and investigations | 39 | 19.5% |
| Foreign Policy | 33 | 16.5% |
| Elections | 31 | 15.5% |
| **Total** | **200** | **100.0%** |

Executive Branch accounted for nearly half of the original dataset and had about 3.13 times as many examples as Elections, the smallest class. The other three categories each had fewer than 40 examples. This gave the model fewer examples from which to learn those categories and left only a small number from each in the validation and test splits. With such small groups, a few changed predictions can noticeably affect per-class scores across random seeds.

### Augmentation and improved class balance

After hyperparameter tuning, I still observed slightly lower accuracy for the Legal and Investigations class and a relatively large performance spread across the three seeds. I first augmented the dataset with 50 additional Legal and Investigations examples and observed a marked improvement. I then added 50 Elections examples and saw further improvement. My best observed accuracy came with a dataset that had less class imbalance.

I used Claude to generate 50 additional synthetic examples for each of the three smaller categories: Legal and Investigations, Foreign Policy, and Elections. I did not add synthetic Executive Branch examples. The current `labels.csv`, which this notebook reads for training and evaluation, therefore contains the original 200 Reddit posts plus 150 Claude-generated synthetic examples, for a total of 350 records.

| Label | Original posts | Added synthetic examples | Current total | Current share |
|---|---:|---:|---:|---:|
| Executive Branch | 97 | 0 | 97 | 27.71% |
| Legal and investigations | 39 | 50 | 89 | 25.43% |
| Foreign Policy | 33 | 50 | 83 | 23.71% |
| Elections | 31 | 50 | 81 | 23.14% |
| **Total** | **200** | **150** | **350** | **100.00%** |

The current class sizes range from 81 to 97 examples, so the largest class is only about 1.20 times the smallest. Executive Branch's share decreased from 48.5% to 27.71%, and each of the other categories now represents roughly one quarter of the dataset. This gives the model more examples of the smaller categories while retaining all 200 original posts. Percentages are rounded.

#### Average confusion matrix — latest test run

Mean counts across seeds **42, 7, and 2024** from the current [results_three_seeds_before.json](results_three_seeds_before.json). This run used the original 200-post dataset, with 30 test examples per seed. Rows are true labels and columns are predictions; values are rounded to two decimal places.

| True label / Predicted label | Executive Branch | Legal and investigations | Foreign Policy | Elections |
|---|---:|---:|---:|---:|
| **Executive Branch** | 12.67 | 0.00 | 1.00 | 0.33 |
| **Legal and investigations** | 3.33 | 1.67 | 0.33 | 0.67 |
| **Foreign Policy** | 0.33 | 0.00 | 4.67 | 0.00 |
| **Elections** | 0.67 | 0.33 | 0.00 | 4.00 |

### Training settings and hyperparameter experiments

For these runs, I used `LEARNING_RATE =  `8.8e-6` and `BATCH_SIZE = 16`. I increased the maximum number of epochs from 3 to `EPOCHS = 25`. Validation and checkpoint saving occur after each epoch, and training stops after two consecutive increases in validation loss. At the end of training, the model restores the checkpoint with the lowest validation loss, so the reported results use the best validation-loss checkpoint. The saved logs show that all three runs completed 15 epochs; 25 was the upper limit.

I also tried `BATCH_SIZE = 8` and observed a slight improvement. Since the homework asked me to focus on one hyperparameter, I retained `BATCH_SIZE = 16` for the reported runs and selected learning rate as the focus of my hyperparameter experiment. The increased epoch limit and early-stopping/checkpoint settings are additional training setup changes documented above.

### Average results across three seeds

These results average seeds 42, 7, and 2024 from [results_three_seeds_before.json](results_three_seeds_before.json), using the augmented 350-record dataset.

**Average accuracy: 86.79%. Average macro F1: 0.8727.** These are the arithmetic means of the three per-seed scores. Each seed used 53 held-out test examples, with an average of 46 correct predictions.

#### Average confusion matrix

Rows are the true labels and columns are the predicted labels. Each cell is the mean count across the three seeds, rounded to two decimal places; fractional counts reflect averaging.

| True label / Predicted label | Executive Branch | Legal and investigations | Foreign Policy | Elections |
|---|---:|---:|---:|---:|
| **Executive Branch** | 13.33 | 1.67 | 0.00 | 0.00 |
| **Legal and investigations** | 2.00 | 10.67 | 0.33 | 0.00 |
| **Foreign Policy** | 1.33 | 0.00 | 11.67 | 0.00 |
| **Elections** | 1.00 | 0.67 | 0.00 | 10.33 |


## 1. Setup

Two cells. Run them once at the start of a session.

The first tells you what hardware you're training on and records it, because
**that is part of your result**. The second confirms the notebook can see your
project files.

In [1]:
# Everything this notebook needs is already in requirements.txt — you installed
# it when you set the project up. This cell just says what you're running on.

import torch


def pick_device():
    """
    Where training will run. Fastest first.

    `cuda`  an NVIDIA GPU.
    `mps`   Apple Silicon's GPU — M1 and later. Much faster than the CPU path.
    `cpu`   everything else. Slower, and it works: this dataset is small enough
            that a full three-seed run is minutes, not hours.
    """
    if torch.cuda.is_available():
        return "cuda", torch.cuda.get_device_name(0)
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps", "Apple Silicon GPU (MPS)"
    return "cpu", "CPU"


DEVICE, DEVICE_NAME = pick_device()

print(f"device: {DEVICE}  ({DEVICE_NAME})")
print(f"torch:  {torch.__version__}")
print()
print("⚠️  WRITE THIS DOWN. Your results depend on it.")
print()
print("Two people running this notebook on the same data with the same seed can")
print("get different numbers, because different hardware does the arithmetic in")
print("a slightly different order. That is not a bug and it is not one of you")
print("being wrong — it is a real property of the thing you are building, and")
print("unit 6 asks you to report it alongside your three seeds.")

device: mps  (Apple Silicon GPU (MPS))
torch:  2.6.0

⚠️  WRITE THIS DOWN. Your results depend on it.

Two people running this notebook on the same data with the same seed can
get different numbers, because different hardware does the arithmetic in
a slightly different order. That is not a bug and it is not one of you
being wrong — it is a real property of the thing you are building, and
unit 6 asks you to report it alongside your three seeds.


### Where you are

Nothing to connect and nothing to clone — this notebook runs inside your own
repo, so `labels.csv` is simply *there* and anything written lands beside it.

**Fill in the two lines.** They only label your results; nothing here talks to
GitHub. You commit what this notebook writes the same way you commit anything
else.

In [2]:
# ── FILL THIS IN ─────────────────────────────────────────────────────────────

NAME  = "Debasish Halder"          # only used to label your results
EMAIL = "dhalder@bu.edu"    # same — nothing here talks to GitHub

# ─────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path

# This notebook runs from inside your own repo, so there is nothing to clone
# and nothing to authenticate. Everything it writes lands next to your code and
# you commit it the same way you commit anything else.
REPO_DIR = Path.cwd()

expected = ["labels.csv", "criteria.md", "baseline.py"]
missing = [f for f in expected if not (REPO_DIR / f).exists()]

if missing and (REPO_DIR / "takemeter.ipynb").exists():
    print(f"Working in: {REPO_DIR}")
    print(f"[note] not found yet: {', '.join(missing)}")
    print("       labels.csv is Milestone 4 — that's expected before then.")
elif missing:
    print("This notebook expects to be run from inside your project folder.")
    print(f"It is currently running in: {REPO_DIR}")
    print("Open the notebook from the repo you cloned, not from a copy.")
else:
    print(f"Working in: {REPO_DIR}")
    print(f"Results will be labelled: {NAME} <{EMAIL}>")
    for f in sorted(os.listdir(REPO_DIR)):
        if not f.startswith("."):
            print(f"  {f}")

Working in: /Users/debasishhalder/projects/Codepath_Main/CodePathWeek5
Results will be labelled: Debasish Halder <dhalder@bu.edu>
  README.md
  RUNNING.md
  __pycache__
  agreement.py
  baseline.py
  baseline_results.json
  classify_neutralpolitics_hybrid.py
  criteria.md
  data
  head
  label_definitions.txt
  labels copy.csv
  labels.csv
  labels_200_rows.csv
  labels_orig.csv
  labels_template.csv
  old_labels.csv
  requirements.txt
  results.json
  results_three_seeds_before.json
  takemeter.ipynb
  takemeter_orig.ipynb
  takemeter_out
  test.py
  test_split.csv


## 2. Your labels

Two things to fill in. Everything after this is automatic.

**`LABELS` must match your CSV exactly** — including capitals and spaces. A
label in your file that isn't in this list is the single most common reason
this notebook fails.

In [ ]:
# ── FILL THIS IN ─────────────────────────────────────────────────────────────

# Your labels, exactly as they appear in the `label` column of your CSV.
#LABELS = ["analysis", "hot_take", "reaction"]
LABELS = ["Executive Branch", "Legal and investigations", "Foreign Policy", "Elections"]

# The model you're fine-tuning. Leave this unless you have a reason.
BASE_MODEL = "distilbert-base-uncased"

# ── Training settings ────────────────────────────────────────────────────────
# These are the defaults. Hyperparameters are vocabulary this unit, not
# decisions — if you change one, write down what and why in your README.
# Epoch changed to 25 as 3 was not doing enough training and making it to 25 with a auto skip with checkpointing once we have get the best validation loss gives us the best model performance  
# Learning rate improved when I reduced and this is the best learning rate that I found after few iterations of tuning and testing the model performance on validation set.

#EPOCHS = 3
EPOCHS = 25 
#LEARNING_RATE = 2e-5
LEARNING_RATE = 8.8e-6
BATCH_SIZE = 16


# How many tokens of each post the model reads. Anything longer is truncated,
# anything shorter is padded up to it — so this is really "how much work per
# post", and it costs time and memory whether your posts are that long or not.
#
# 128 covers almost every post in a corpus like this. Raising it to 256 roughly
# doubles both the time and the memory for text most people don't have.
MAX_LENGTH = 128

# ── The seed ─────────────────────────────────────────────────────────────────
# This fixes which posts land in which split. Change it and you get a different
# split and different numbers.
#
# ⚠️ NEXT UNIT YOU CHANGE THIS THREE TIMES. That's not busywork — it's how you
# find out whether your number is real or whether you got a friendly draw.

SEED = 42

print(f"{len(LABELS)} labels: {', '.join(LABELS)}")
print(f"seed {SEED}, {EPOCHS} epochs, lr {LEARNING_RATE}, batch {BATCH_SIZE}")

4 labels: Executive Branch, Legal and investigations, Foreign Policy, Elections
seed 42, 25 epochs, lr 8.8e-06, batch 16


### Your data

`labels.csv` came down with the clone, so there's nothing to upload. If you've
just added rows on your laptop, commit and push them there first, then re-run
the connect cell.

Don't split the file yourself — the next section does that, and doing it twice
is how examples leak between training and test.

No data of your own yet? Point `CSV` at `data/practice_labels.csv`, the 60-post
set used in class.

In [4]:
import pandas as pd

CSV = "./labels.csv"              # "data/practice_labels.csv" to practise

df = pd.read_csv(CSV)

print(f"Loaded {CSV}: {len(df)} rows")
print(f"Columns: {', '.join(df.columns)}")

Loaded ./labels.csv: 350 rows
Columns: text, label, note


### Check it before you train

This catches the problems that would otherwise show up as a confusing error
forty minutes from now — or, worse, as a model that trained on bad data and
told you nothing was wrong.

In [5]:
def check_dataset(df, labels):
    """
    Look for the things that break a training run or quietly ruin it.

    Returns (problems, warnings). Problems stop you; warnings are worth reading.
    """
    problems, warnings = [], []

    for column in ("text", "label"):
        if column not in df.columns:
            problems.append(f"No '{column}' column. Found: {', '.join(df.columns)}")
    if problems:
        return problems, warnings

    blank = df["text"].isna().sum() + (df["text"].astype(str).str.strip() == "").sum()
    if blank:
        problems.append(f"{blank} rows have an empty text column.")

    found = set(df["label"].dropna().astype(str).str.strip().unique())
    expected = set(labels)

    unknown = found - expected
    if unknown:
        problems.append(
            f"Labels in your CSV that aren't in LABELS: {sorted(unknown)}. "
            f"Check capitals and spaces — 'Hot_take' and 'hot_take' are different."
        )

    unused = expected - found
    if unused:
        warnings.append(f"Labels in LABELS that never appear in your CSV: {sorted(unused)}")

    counts = df["label"].value_counts()
    total = len(df)
    for label, count in counts.items():
        share = count / total
        if share > 0.70:
            problems.append(
                f"'{label}' is {share:.0%} of your data. The brief caps this at 70% — "
                f"above that the model learns to guess it. Collect more of the thin labels."
            )
        elif share > 0.60:
            warnings.append(f"'{label}' is {share:.0%} of your data. Getting lopsided.")

    if total < 150:
        warnings.append(
            f"{total} rows. The brief's floor for a submission is 150 and it asks "
            f"for 200. Training will still run — this is fine for the practice "
            f"data, and something to fix before you submit."
        )
    elif total < 200:
        warnings.append(f"{total} rows — under the 200 the brief asks for. Say so in your README.")

    duplicates = df["text"].duplicated().sum()
    if duplicates:
        warnings.append(
            f"{duplicates} duplicate posts. They'll land in different splits and "
            f"inflate your score — that's leakage."
        )

    for label, count in counts.items():
        if count < 20:
            warnings.append(
                f"Only {count} examples of '{label}'. Its test split will be tiny, "
                f"so its F1 will jump around between seeds."
            )

    return problems, warnings


problems, warnings = check_dataset(df, LABELS)

for w in warnings:
    print(f"[look at]  {w}")
for p in problems:
    print(f"[FIX ME]   {p}")

if not problems:
    print("\nData looks fine.")
    print()
    print(df["label"].value_counts().to_string())
else:
    print("\nFix the [FIX ME] lines before training.")


Data looks fine.

label
Executive Branch            97
Legal and investigations    89
Foreign Policy              83
Elections                   81


## 3. Split

70% train, 15% validation, 15% test.

The split is **stratified** — each split gets roughly the same label proportions
as the whole file. Without that, a small label can end up with nothing in the
test set and its F1 becomes undefined.

**Lock the test set and don't look at it again until evaluation.** Checking how
you're doing against the test set and then changing things is the slow-motion
version of training on it.

In [6]:
from sklearn.model_selection import train_test_split


def split_dataset(df, seed, test_size=0.15, val_size=0.15):
    """
    Stratified 70/15/15. Returns (train, val, test) DataFrames.

    Stratifying falls back to a plain random split if any label is too rare to
    stratify on — that's a warning sign about the dataset, not about the split.
    """
    def _stratify_on(frame):
        counts = frame["label"].value_counts()
        return frame["label"] if counts.min() >= 2 else None

    rest, test = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=_stratify_on(df)
    )
    val_share = val_size / (1 - test_size)
    train, val = train_test_split(
        rest, test_size=val_share, random_state=seed, stratify=_stratify_on(rest)
    )
    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )


def describe_splits(train, val, test, labels):
    """A table of per-label counts per split, and a note on the thin ones."""
    import pandas as pd

    table = pd.DataFrame({
        "train": train["label"].value_counts(),
        "val": val["label"].value_counts(),
        "test": test["label"].value_counts(),
    }).reindex(labels).fillna(0).astype(int)

    thin = [label for label in labels if table.loc[label, "test"] < 8]
    return table, thin


train_df, val_df, test_df = split_dataset(df, SEED)
table, thin = describe_splits(train_df, val_df, test_df, LABELS)

print(f"train {len(train_df)}  ·  val {len(val_df)}  ·  test {len(test_df)}\n")
print(table.to_string())

if thin:
    print()
    for label in thin:
        n = table.loc[label, "test"]
        print(f"[note] '{label}' has only {n} examples in the test split.")
    print()
    print("A metric computed on that few examples moves a lot between seeds.")
    print("Write that down now — next unit it's a diagnosis, not a surprise.")

train 244  ·  val 53  ·  test 53

                          train  val  test
label                                     
Executive Branch             67   15    15
Legal and investigations     62   14    13
Foreign Policy               58   12    13
Elections                    57   12    12


## 4. Train

About ten minutes on a T4. It'll print a loss for each epoch as it goes.

**The loss curve tells you training is converging — not that the model learned
what you meant.** A model can drive loss down by memorising or by latching onto
something spurious, and neither is visible here. That's what next unit is for.

In [7]:
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

SKIP_TRAIN_OLD = True  # Set to False to run the old training.

if not SKIP_TRAIN_OLD:
    def to_ids(df, labels):
        """Map label strings to the integer ids the model wants."""
        lookup = {label: i for i, label in enumerate(labels)}
        out = df.copy()
        out["labels"] = out["label"].astype(str).str.strip().map(lookup)
        return out[["text", "labels"]]


    def compute_metrics(eval_pred):
        logits, gold = eval_pred
        predicted = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(gold, predicted),
            "f1_macro": f1_score(gold, predicted, average="macro", zero_division=0),
        }


    def train_model(train_df, val_df, labels, seed, epochs, learning_rate, batch_size,
                    base_model, quiet=False):
        """Fine-tune and return (trainer, tokenizer)."""
        set_seed(seed)

        tokenizer = AutoTokenizer.from_pretrained(base_model)
        model = AutoModelForSequenceClassification.from_pretrained(
            base_model,
            num_labels=len(labels),
            id2label={i: l for i, l in enumerate(labels)},
            label2id={l: i for i, l in enumerate(labels)},
        )

        def tokenize(batch):
            return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

        train_ds = Dataset.from_pandas(to_ids(train_df, labels)).map(tokenize, batched=True)
        val_ds = Dataset.from_pandas(to_ids(val_df, labels)).map(tokenize, batched=True)

        args = TrainingArguments(
            output_dir="./takemeter_out",
            num_train_epochs=epochs,
            learning_rate=learning_rate,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            eval_strategy="epoch",
            save_strategy="no",
            logging_steps=10,
            seed=seed,
            report_to="none",
            disable_tqdm=quiet,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_metrics,
        )
        trainer.train()
        return trainer, tokenizer


    trainer, tokenizer = train_model(
        train_df, val_df, LABELS, SEED, EPOCHS, LEARNING_RATE, BATCH_SIZE, BASE_MODEL
    )
    print("\nTrained.")
else:
    print("Skipped old training. Run the early-stopping cell next.")


Skipped old training. Run the early-stopping cell next.


In [8]:
# Early-stopping training: use this cell instead of the preceding training cell.
# Run Sections 1-3 first, then this cell, then Section 5 (Results).
# Section 6 will use this train_model function for its three-seed runs.

from uuid import uuid4
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, TrainerCallback, set_seed,
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support


def to_ids(df, labels):
    """Map label strings to the integer ids the model wants."""
    lookup = {label: i for i, label in enumerate(labels)}
    out = df.copy()
    out["labels"] = out["label"].astype(str).str.strip().map(lookup)
    return out[["text", "labels"]]


def compute_metrics(eval_pred):
    logits, gold = eval_pred
    predicted = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(gold, predicted),
        "f1_macro": f1_score(gold, predicted, average="macro", zero_division=0),
    }


class StopAfterTwoLossIncreases(TrainerCallback):
    def __init__(self):
        self.previous_loss = None
        self.increases = 0

    def on_train_begin(self, args, state, control, **kwargs):
        self.previous_loss = None
        self.increases = 0
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None or "eval_loss" not in metrics:
            return control

        current_loss = float(metrics["eval_loss"])
        if self.previous_loss is not None and current_loss > self.previous_loss:
            self.increases += 1
        else:
            self.increases = 0

        self.previous_loss = current_loss
        if self.increases >= 2:
            control.should_training_stop = True
        return control


def train_model(train_df, val_df, labels, seed, epochs, learning_rate,
                batch_size, base_model, quiet=False):
    """Train up to epochs, stop after two loss rises, and restore the best model."""
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=len(labels),
        id2label={i: label for i, label in enumerate(labels)},
        label2id={label: i for i, label in enumerate(labels)},
    )

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    train_ds = Dataset.from_pandas(
        to_ids(train_df, labels)
    ).map(tokenize, batched=True)
    val_ds = Dataset.from_pandas(
        to_ids(val_df, labels)
    ).map(tokenize, batched=True)

    # Separate checkpoints for each run, including repeated runs of one seed.
    run_id = uuid4().hex[:8]
    args = TrainingArguments(
        output_dir=f"./takemeter_out/seed_{seed}_{run_id}",
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        logging_steps=10,
        seed=seed,
        report_to="none",
        disable_tqdm=quiet,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[StopAfterTwoLossIncreases()],
    )
    trainer.train()

    # load_best_model_at_end has already restored the lowest-loss checkpoint.
    print(f"Completed epochs: {trainer.state.epoch:.0f}")
    print(f"Best validation loss: {trainer.state.best_metric:.6f}")
    print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
    return trainer, tokenizer


trainer, tokenizer = train_model(
    train_df, val_df, LABELS, SEED,
    EPOCHS, LEARNING_RATE, BATCH_SIZE, BASE_MODEL
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/244 [00:00<?, ? examples/s]

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.379500,1.337112,0.641509,0.648120
2,1.293576,1.218531,0.773585,0.791511
3,1.199134,1.058843,0.811321,0.824890
4,0.981296,0.893595,0.792453,0.808255
5,0.766450,0.755016,0.811321,0.827156
6,0.645962,0.665182,0.830189,0.843285
7,0.523459,0.600733,0.830189,0.843285
8,0.429960,0.559193,0.811321,0.826317
9,0.369787,0.529445,0.811321,0.826793
10,0.317004,0.515952,0.830189,0.842765


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Completed epochs: 15
Best validation loss: 0.498966
Best checkpoint: ./takemeter_out/seed_42_3e95a0fb/checkpoint-176


## 5. Results

Runs the model on the test split — the one it has never seen — and writes
`results.json`.

`results.json` records **how sure the model was**, whether its confident
guesses were right more often than its unsure ones. If one of your criteria is
about confidence, that's the number that tests it. It also records **the device
you trained on**, which you'll want next unit when your numbers and someone
else's don't match.

It also writes `test_split.csv`, the held-out posts themselves. Next unit
`baseline.py` reads that file so the baseline is scored on exactly these posts
and not a different sample. Two numbers from two different sets of posts aren't
a comparison.

**Commit both files.** Don't read the numbers closely yet — your criteria are
already filed, and reading results before you've written your standard is how
the standard quietly moves.

In [9]:
import json


def evaluate(trainer, tokenizer, test_df, labels, seed=None):
    """Score the model on the held-out split. Returns a plain dict."""
    from datasets import Dataset

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = Dataset.from_pandas(to_ids(test_df, labels)).map(tokenize, batched=True)
    output = trainer.predict(test_ds)

    predicted = np.argmax(output.predictions, axis=-1)
    gold = np.array(output.label_ids)

    # How sure the model was about each guess. Softmax over the raw scores, then
    # the winning one. A criterion about confidence can't be tested without it.
    scores = output.predictions
    exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
    confidence = (exp / exp.sum(axis=-1, keepdims=True)).max(axis=-1)

    precision, recall, f1, support = precision_recall_fscore_support(
        gold, predicted, labels=list(range(len(labels))), zero_division=0
    )

    return {
        "seed": seed,
        "n_test": int(len(gold)),
        "accuracy": float(accuracy_score(gold, predicted)),
        "f1_macro": float(f1_score(gold, predicted, average="macro", zero_division=0)),
        "per_label": {
            label: {
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
            }
            for i, label in enumerate(labels)
        },
        "confusion": confusion_counts(gold, predicted, labels),
        "confidence": confidence_report(confidence, gold, predicted),
    }


def confidence_report(confidence, gold, predicted):
    """
    Are the model's confident guesses right more often than its unsure ones?

    If you wrote a criterion about confidence, this is what tests it. If the
    two thirds score about the same, the model's confidence means nothing —
    which is itself worth writing down.
    """
    right = gold == predicted
    ranked = np.argsort(confidence)
    third = max(1, len(confidence) // 3)
    least, most = ranked[:third], ranked[-third:]

    return {
        "mean_when_right": float(confidence[right].mean()) if right.any() else None,
        "mean_when_wrong": float(confidence[~right].mean()) if (~right).any() else None,
        "accuracy_most_confident_third": float(right[most].mean()),
        "accuracy_least_confident_third": float(right[least].mean()),
        "n_per_third": int(third),
    }


def confusion_counts(gold, predicted, labels):
    """Rows are true labels, columns are guesses. Plain nested dict."""
    matrix = {t: {p: 0 for p in labels} for t in labels}
    for g, p in zip(gold, predicted):
        matrix[labels[int(g)]][labels[int(p)]] += 1
    return matrix


results = evaluate(trainer, tokenizer, test_df, LABELS, seed=SEED)

# The device goes in the file, not just the console. Next unit you compare your
# numbers against your own other seeds and against other people's — and you
# cannot read a difference you can't attribute.
results["device"] = DEVICE
results["device_name"] = DEVICE_NAME
results["torch_version"] = torch.__version__
results["max_length"] = MAX_LENGTH

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

# The held-out posts themselves, for next unit's baseline to score.
test_df.to_csv("test_split.csv", index=False)

print(f"accuracy  {results['accuracy']:.3f}")
print(f"macro F1  {results['f1_macro']:.3f}\n")
for label, scores in results["per_label"].items():
    print(f"  {label:<16} F1 {scores['f1']:.3f}   (n={scores['support']})")

c = results["confidence"]
print(f"\nconfidence   most-sure third {c['accuracy_most_confident_third']:.3f}"
      f"   ·   least-sure third {c['accuracy_least_confident_third']:.3f}")

print()

print("Wrote results.json and test_split.csv. Commit them both — they're the")
print("evidence the run happened, and next unit reads test_split.csv.")


Map:   0%|          | 0/53 [00:00<?, ? examples/s]

accuracy  0.830
macro F1  0.837

  Executive Branch F1 0.750   (n=15)
  Legal and investigations F1 0.720   (n=13)
  Foreign Policy   F1 0.923   (n=13)
  Elections        F1 0.957   (n=12)

confidence   most-sure third 1.000   ·   least-sure third 0.588

Wrote results.json and test_split.csv. Commit them both — they're the
evidence the run happened, and next unit reads test_split.csv.


---
---

# ⬇️ Unit 6 starts here

Nothing below this line is needed in unit 5. Come back next unit.

---

## 6. Three seeds

Your score depends on which posts landed in the test split. Change the split,
change the score. So run it three times.

**This is one cell on purpose** — all three seeds run in a single execution, so
you get a comparable set or none at all. On a laptop CPU it's a few minutes per
seed; start it and go and do Milestone 3 while it runs.

If accuracy swings more than about ten points across the three, your dataset is
too small or too lopsided for a stable measure. That's a genuine diagnosis, not
a mistake.

**Set `RUN_LABEL` before each run.** `"before"` now; `"after"` when you come
back in Milestone 5. Leave it and the improvement overwrites the numbers you
are comparing it against.

In [10]:
SEEDS = [42, 7, 2024]

# WARNING: change this to "after" before you re-run this cell in Milestone 5.
#
# Without it the improvement run overwrites your before-numbers, and the file
# you committed as evidence quietly becomes the other set. Same idea as the
# --label flag on the eval scripts in units 2 and 4.
RUN_LABEL = "before"

all_results = []

for i, seed in enumerate(SEEDS, 1):
    print(f"\n{'=' * 60}\nSeed {seed}  ({i} of {len(SEEDS)})\n{'=' * 60}")

    tr, va, te = split_dataset(df, seed)
    t, tok = train_model(
        tr, va, LABELS, seed, EPOCHS, LEARNING_RATE, BATCH_SIZE, BASE_MODEL, quiet=True
    )
    r = evaluate(t, tok, te, LABELS, seed=seed)
    all_results.append(r)

    print(f"  accuracy {r['accuracy']:.3f}   macro F1 {r['f1_macro']:.3f}")

OUT = f"results_three_seeds_{RUN_LABEL}.json"

for r in all_results:
    r["device"] = DEVICE
    r["device_name"] = DEVICE_NAME
    r["torch_version"] = torch.__version__
    r["max_length"] = MAX_LENGTH

with open(OUT, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\n\n{'=' * 60}\nAll three done.\n{'=' * 60}")
print(f"Wrote {OUT}.")
if RUN_LABEL == "before":
    print('Re-running after your improvement? Set RUN_LABEL = \"after" first,')
    print('or you will overwrite the numbers you are comparing against.')

print(f"Wrote {OUT}. Commit it.")



Seed 42  (1 of 3)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/244 [00:00<?, ? examples/s]

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '30.3', 'train_samples_per_second': '201.3', 'train_steps_per_second': '13.2', 'train_loss': '0.5959', 'epoch': '15'}
Completed epochs: 15
Best validation loss: 0.461161
Best checkpoint: ./takemeter_out/seed_2024_89385d46/checkpoint-208


Map:   0%|          | 0/53 [00:00<?, ? examples/s]

  accuracy 0.887   macro F1 0.889


All three done.
Wrote results_three_seeds_before.json.
Re-running after your improvement? Set RUN_LABEL = "after" first,
or you will overwrite the numbers you are comparing against.
Wrote results_three_seeds_before.json. Commit it.


### The numbers, as a table

Paste this into your README's run log. The Target and Verdict columns are yours
to fill in from `criteria.md` — the notebook doesn't know what you promised.

In [11]:
def spread_table(all_results, labels):
    """A markdown table of every measure across every seed."""
    seeds = [r["seed"] for r in all_results]
    header = f"| Measure | {' | '.join(f'Seed {s}' for s in seeds)} | Spread |"
    divider = "|---|" + "|".join(["---"] * (len(seeds) + 1)) + "|"
    rows = [header, divider]

    def row(name, values, fmt="{:.3f}"):
        spread = max(values) - min(values)
        cells = " | ".join(fmt.format(v) for v in values)
        return f"| {name} | {cells} | {fmt.format(spread)} |"

    rows.append(row("Overall accuracy", [r["accuracy"] for r in all_results]))
    rows.append(row("Macro F1", [r["f1_macro"] for r in all_results]))
    for label in labels:
        rows.append(row(f"F1 — `{label}`",
                        [r["per_label"][label]["f1"] for r in all_results]))
    return "\n".join(rows)


def stability_note(all_results):
    """Say plainly whether the spread makes these numbers trustworthy."""
    accuracies = [r["accuracy"] for r in all_results]
    spread = max(accuracies) - min(accuracies)
    if spread > 0.10:
        return (f"Accuracy moved {spread:.3f} across seeds — more than 10 points. "
                f"Your dataset is too small or too lopsided for a stable measure. "
                f"That IS a diagnosis. Write it down.")
    if spread > 0.05:
        return (f"Accuracy moved {spread:.3f} across seeds. Enough that a criterion "
                f"landing within that band can't really be called either way.")
    return f"Accuracy moved {spread:.3f} across seeds. Stable enough to trust."


print(spread_table(all_results, LABELS))
print()
print(stability_note(all_results))

| Measure | Seed 42 | Seed 7 | Seed 2024 | Spread |
|---|---|---|---|---|
| Overall accuracy | 0.830 | 0.887 | 0.887 | 0.057 |
| Macro F1 | 0.837 | 0.892 | 0.889 | 0.055 |
| F1 — `Executive Branch` | 0.750 | 0.839 | 0.857 | 0.107 |
| F1 — `Legal and investigations` | 0.720 | 0.857 | 0.880 | 0.160 |
| F1 — `Foreign Policy` | 0.923 | 0.917 | 0.960 | 0.043 |
| F1 — `Elections` | 0.957 | 0.957 | 0.857 | 0.099 |

Accuracy moved 0.057 across seeds. Enough that a criterion landing within that band can't really be called either way.


## 7. Confusion matrix

Rows are true labels, columns are the model's guesses. The diagonal is right
answers; everything off it is a mistake with a direction.

**Your README needs this typed out as a markdown table.** The cell prints one
you can copy. An image of a matrix earns nothing.

Then find your biggest off-diagonal number and say what it means in one
sentence. Not "the model made mistakes" — *which* boundary it didn't learn, and
which way round.

In [12]:
def matrix_markdown(confusion, labels):
    """The confusion matrix as a markdown table, ready to paste."""
    header = f"| true \\ predicted | {' | '.join(labels)} |"
    divider = "|---|" + "|".join(["---"] * len(labels)) + "|"
    rows = [header, divider]
    for true_label in labels:
        cells = " | ".join(str(confusion[true_label][p]) for p in labels)
        rows.append(f"| **{true_label}** | {cells} |")
    return "\n".join(rows)


def biggest_confusion(confusion, labels):
    """The largest off-diagonal cell: (true, predicted, count)."""
    worst = None
    for true_label in labels:
        for predicted in labels:
            if true_label == predicted:
                continue
            count = confusion[true_label][predicted]
            if worst is None or count > worst[2]:
                worst = (true_label, predicted, count)
    return worst


confusion = all_results[0]["confusion"]

print(matrix_markdown(confusion, LABELS))
print()

true_label, predicted, count = biggest_confusion(confusion, LABELS)
reverse = confusion[predicted][true_label]

print(f"Biggest confusion: {count} real `{true_label}` posts called `{predicted}`.")
print(f"The other direction: {reverse} real `{predicted}` posts called `{true_label}`.")
print()
if count > reverse * 2 and count > 2:
    print(f"That's lopsided — the model leans toward `{predicted}` on this pair.")
    print(f"A direction like that usually points at your labelling rather than the")
    print(f"model. Your agreement report is the thing that can tell you which.")
else:
    print("Roughly symmetric — the model finds this pair hard in both directions,")
    print("which more often means the boundary itself is genuinely hard.")

| true \ predicted | Executive Branch | Legal and investigations | Foreign Policy | Elections |
|---|---|---|---|---|
| **Executive Branch** | 12 | 3 | 0 | 0 |
| **Legal and investigations** | 3 | 9 | 1 | 0 |
| **Foreign Policy** | 1 | 0 | 12 | 0 |
| **Elections** | 1 | 0 | 0 | 11 |

Biggest confusion: 3 real `Executive Branch` posts called `Legal and investigations`.
The other direction: 3 real `Legal and investigations` posts called `Executive Branch`.

Roughly symmetric — the model finds this pair hard in both directions,
which more often means the boundary itself is genuinely hard.


---

### Before you close this notebook

- [ ] `results_three_seeds.json` pushed (the last cell does it — check it said so)
- [ ] The spread table pasted into your README
- [ ] The confusion matrix **typed as a markdown table** in your README
- [ ] One sentence on your biggest off-diagonal number

Then go and do the agreement check against the staff set. It's the only
instrument you have that can tell a labelling problem from a model problem —
and Milestone 4 asks you to use it as exactly that.